<a href="https://colab.research.google.com/github/isismeira/natural_language_processing/blob/main/parsing_llms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git

In [ ]:
!pip install datasets==3.6.0

In [1]:
from datasets import load_dataset
import pandas as pd

In [2]:
ds = load_dataset("matejklemen/vuamc")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


0000.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16740 [00:00<?, ? examples/s]

In [3]:
df = ds["train"].to_pandas()

In [4]:
df.head()

,document_name,words,pos_tags,met_type,meta
0,a1e-fragment01,"[Latest, corporate, unbundler, reveals, laid-b...","[AJS, AJ0, NN1, VVZ, AJ0, NN1, PUN, NP0, NP0, ...","[{'type': 'mrw/met', 'word_indices': [3]}, {'t...","[N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, ..."
1,a1e-fragment01,"[By, FRANK, KANE]","[PRP, NP0, NP0-NN1]",[],"[N/A, N/A, N/A]"
2,a1e-fragment01,"[IT, SEEMS, that, Roland, Franklin, ,, the, la...","[PNP, VVZ, CJT, NP0, NP0, PUN, AT0, AJS, NN1, ...","[{'type': 'mrw/met', 'word_indices': [16]}, {'...","[N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, ..."
3,a1e-fragment01,"[He, has, not, properly, investigated, the, ta...","[PNP, VHZ, XX0, AV0, VVN, AT0, NN1, POS, NN1, ...","[{'type': 'mrw/met', 'word_indices': [6]}]","[N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, ..."
4,a1e-fragment01,"[The, 63-year-old, head, of, Pembridge, Invest...","[AT0, AJ0, NN1, PRF, NP0, NN2, PUN, PRP, DTQ, ...","[{'type': 'mrw/met', 'word_indices': [2]}, {'t...","[N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, ..."


In [5]:
palavras = df.iloc[0, 1]
print(palavras)
type(palavras)

['Latest' 'corporate' 'unbundler' 'reveals' 'laid-back' 'approach' ':'
 'Roland' 'Franklin' ',' 'who' 'is' 'leading' 'a' '697m' 'pound'
 'break-up' 'bid' 'for' 'DRG' ',' 'talks' 'to' 'Frank' 'Kane']


numpy.ndarray

In [7]:
# Use o método join() para concatenar os elementos do array 'palavras'
# O espaço ' ' entre as aspas é o separador que será usado entre os elementos.
string_concatenada = " ".join(palavras)

# Imprima a string concatenada
print(string_concatenada)

# Verifique o tipo da nova variável
print(type(string_concatenada))

Latest corporate unbundler reveals laid-back approach : Roland Franklin , who is leading a 697m pound break-up bid for DRG , talks to Frank Kane
<class 'str'>


In [10]:

!pip install -q google-generativeai

import os
import re
import ast
import google.generativeai as genai
from google.colab import userdata

# 1. Ler a chave da variável de ambiente
api_key = userdata.get('GEMINI_TOKEN')
if not api_key:
    raise ValueError("A variável de ambiente GEMINI_TOKEN não foi definida.")

# 2. Configurar o cliente
genai.configure(api_key=api_key)

# 3. Função para extrair unidades metafóricas
def extrair_unidades_metaforicas(string_concatenada: str):
    prompt = (
        f"Retorne SOMENTE as palavras que são unidades metafóricas na frase: {string_concatenada}. "
        f"Responda exclusivamente em formato de lista Python válida, por exemplo: ['x', 'y'] "
        f"Não escreva explicações, não inclua código, não inclua texto fora da lista."
    )

    # Chamar o modelo
    response = genai.GenerativeModel("gemini-2.0-flash-lite").generate_content(prompt)

    # Extrair texto de saída
    resultado = response.text.strip()

    # Tentar extrair lista com regex
    match = re.search(r"\[.*?\]", resultado, re.DOTALL)
    if not match:
        raise ValueError(f"Nenhuma lista Python encontrada na resposta: {resultado}")

    lista_str = match.group(0)

    try:
        # Usar ast.literal_eval para segurança
        lista_metaforas = ast.literal_eval(lista_str)
        if isinstance(lista_metaforas, list):
            return lista_metaforas
        else:
            raise ValueError("O retorno não é uma lista Python.")
    except Exception as e:
        raise ValueError(f"Erro ao interpretar a lista: {lista_str}") from e


metaforas = extrair_unidades_metaforicas(string_concatenada)
print(metaforas)


['unbundler', 'laid-back', 'break-up']
